# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset for protein abundance and peptide sequence analysis using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

# Display metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s so you know what data you can extract.

In [ ]:
# List available record sets with their @id and name
record_sets = list(ds.metadata.record_sets)
print('Available record sets:')
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {getattr(rs, 'name', '')}")
    # Show fields for each record set
    print("  Fields:")
    for field in rs.fields:
        colinfo = f"  (col @id: {field.column.id if getattr(field, 'column', None) else ''})" if getattr(field, 'column', None) else ''
        print(f"    - @id: {field.id} | name: {getattr(field, 'name', '')} | type: {getattr(field, 'data_type', '')} {colinfo}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Reference record set and field `@id`s from the overview above.

In [ ]:
# Gather record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(ds.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Extracted columns: {df.columns.tolist()}")
        display(df.head(2))
    else:
        print("  No records found.")

# For analysis, pick the main record set (the largest table)
main_rs_id = max(dataframes, key=lambda x: len(dataframes[x]) if x in dataframes else 0)
print(f"\nMain record set '@id' for analysis: {main_rs_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, like filtering records by a numeric attribute, normalizing, or aggregating by a group key. We'll demonstrate on the main protein data table.

In [ ]:
# Choose a numeric field for demonstration
# To find available numeric fields, display columns of the main table
df = dataframes[main_rs_id]
print("Main table columns:", df.columns.tolist())

# Let's try to pick a numeric field (common in proteomics: e.g. Coverage, PeptideCounts, MW, pI, etc.)
possible_numeric = [col for col in df.columns if df[col].dtype in ["int64", "float64"] or pd.api.types.is_numeric_dtype(df[col])]
if possible_numeric:
    numeric_field = possible_numeric[0]  # Pick the first numeric column
else:
    # If all fields are strings, try force-convert one likely to be numeric (by examining name)
    likely_numerics = [c for c in df.columns if any(k in c.lower() for k in ["coverage", "count", "mw", "pi", "abundance"])]
    numeric_field = likely_numerics[0] if likely_numerics else df.columns[0]
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    print(f"Forcing numeric conversion for field: {numeric_field}")

print(f"Chosen numeric field: {numeric_field}")

# Filter by this field (e.g. entries where numeric_field > threshold)
threshold = df[numeric_field].dropna().quantile(0.75)  # Use 75th percentile as threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
display(filtered_df.head(3))

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))

# Try to group by a field, e.g. a protein property (check common protein grouping fields)
group_field = None
candidate_groups = ["accession", "ProteinAccession", "ProteinID", "description", "group", "sample"]
for g in candidate_groups:
    for c in df.columns:
        if g.lower() in c.lower():
            group_field = c
            break
    if group_field:
        break

if group_field:
    print(f"Grouping by '{group_field}' and aggregating mean {numeric_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    display(grouped_df.head(3))
else:
    print(f"No suitable group field found in main table columns: {df.columns.tolist()}")

## 5. Visualization
Visualize the distribution and possible relationships between fields, e.g., distribution of abundance or counts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If another numeric field exists, scatterplot
other_numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != numeric_field]
if other_numeric:
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=df[numeric_field], y=df[other_numeric[0]])
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric[0])
    plt.title(f"{numeric_field} vs {other_numeric[0]}")
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the Mass Spectrometry dataset with mlcroissant, listed and examined its record sets and fields (referenced by their `@id`), extracted and analyzed the protein records, filtered and normalized numeric attributes, grouped by identifiers, and visualized core distributions.

Further biological or statistical analysis and machine learning can follow, now that the FAIR² dataset is easily available as pandas DataFrames, with each field referenced by its Croissant `@id` for provenance.